# Step 5 — Climate Projection Panel (LOCA2)

**Objective:** Project future extreme heat for ERCOT and CAISO under SSP2-4.5 and SSP5-8.5 using USGS CMIP6-LOCA2 county summaries (27 GCMs, 1950-2100).

**Data source:** https://data.usgs.gov/datacatalog/data/USGS:673d0719d34e6b795de6b593

**Output:** `data/processed/loca2_projections_ercot.csv` and `loca2_projections_caiso.csv`

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from config.settings import PROCESSED, ERCOT_FIPS, CAISO_FIPS, PROJ_PERIODS, SSP_SCENARIOS
from src.data.loca2 import (
    load_loca2, compute_climatology, compute_change_factors,
    build_ercot_projections, build_caiso_projections, LOCA2_VARIABLES
)
from src.viz.maps import projection_bar


## 5.1 Available LOCA2 Climdex variables

In [ ]:
for var, desc in LOCA2_VARIABLES.items():
    print(f'  {var:10s}: {desc}')


## 5.2 Build ERCOT projection panel

In [ ]:
try:
    ercot_proj = build_ercot_projections()
    print(f'ERCOT projections shape: {ercot_proj.shape}')
    print(ercot_proj.head())
except (FileNotFoundError, RuntimeError) as e:
    print(f'LOCA2 data not yet downloaded:\n{e}')
    ercot_proj = None


## 5.3 Build CAISO projection panel

In [ ]:
try:
    caiso_proj = build_caiso_projections()
    print(f'CAISO projections shape: {caiso_proj.shape}')
except (FileNotFoundError, RuntimeError) as e:
    print(f'LOCA2 data not yet downloaded:\n{e}')
    caiso_proj = None


## 5.4 Key results: additional summer days and heatwave duration by 2050

In [ ]:
if ercot_proj is not None:
    mid_ssp585 = ercot_proj[
        (ercot_proj['scenario'] == 'ssp585') &
        (ercot_proj['period_label'] == 'mid')
    ]
    for metric in ['SU_delta', 'WSDI_delta', 'TR_delta', 'TXx_delta']:
        if metric in mid_ssp585.columns:
            print(f'\n{metric} (SSP5-8.5, mid-century):')
            print(mid_ssp585[metric].describe())


## 5.5 Visualise projected WSDI change across ERCOT counties

In [ ]:
if ercot_proj is not None and 'WSDI_delta' in ercot_proj.columns:
    fig, ax = plt.subplots(figsize=(14, 6))
    projection_bar(
        ercot_proj[ercot_proj['period_label'] == 'mid'],
        metric_delta='WSDI_delta',
        title='Projected change in Warm Spell Duration Index — ERCOT (mid-century)',
        ax=ax,
    )
    plt.tight_layout()
    plt.savefig('../data/processed/ercot_wsdi_delta.png', dpi=150)
    plt.show()


## 5.6 Save projection panels

In [ ]:
if ercot_proj is not None:
    ercot_proj.to_csv(PROCESSED['loca2_ercot'], index=False)
    print('Saved:', PROCESSED['loca2_ercot'])
if caiso_proj is not None:
    caiso_proj.to_csv(PROCESSED['loca2_caiso'], index=False)
    print('Saved:', PROCESSED['loca2_caiso'])
